In [ ]:
Microservice architecture is an architectural style where an application is built
as a collection of small, independent services, each responsible for a single business
capability, communication over lightweight protocols. (HTTPS / events)

Each service:

    Has its own database
    Is independently deployable
    Scales independently
    Owned by a small team

In [ ]:
👉 Kafka is used for EVENT-DRIVEN COMMUNICATION

Why NOT REST everywhere?

Tight coupling

Poor scalability

In [ ]:
Loose coupling
✔ Async processing
✔ High throughput
✔ Replay events
✔ Scales well

6️⃣ Kafka Topic Example

In [ ]:
❓ Q1: What happens if Payment fails?

Answer (Industry Standard)

    Payment Service emits PAYMENT_FAILED
    Order Service updates status → FAILED
    Inventory Service releases stock
    Notification Service informs user

In [ ]:
Q3: What if Kafka is down?

Answer

Services use retry
Events stored in local DB (outbox pattern)
Kafka resumes → events replayed

In [ ]:
Q4: How do services communicate?

Answer

    Synchronous: REST (User → Order)
    Asynchronous: Kafka (Order → Inventory)

In [ ]:
9️⃣ API Gateway Role

Responsibilities

    Authentication
    Rate limiting
    Routing
    Logging
    Tools
    Kong
    Nginx
    AWS API Gateway

In [ ]:
| Service   | DB         |
| --------- | ---------- |
| User      | MySQL      |
| Product   | MongoDB    |
| Cart      | Redis      |
| Order     | PostgreSQL |
| Inventory | MySQL      |


In [ ]:
API Gateway
   |
-----------------------------------------
| User | Product | Cart | Order | Payment |
-----------------------------------------
          |
      Inventory
          |
      Notification
          |
        Kafka


In [ ]:
3️⃣ Core Services Explained (Interview Level)

1️⃣ User Service

Responsibility

    Register / Login
    JWT authentication
    User profile, address

POST /users/register
POST /users/login
GET  /users/{id}

Database

User DB (MySQL / PostgreSQL)

In [ ]:
2️⃣ Product Service

Responsibility
Product catalog
Price, description, images
Search & filter

Database
MongoDB (document-based works well)

GET /products
GET /products/{id}

In [ ]:
3️⃣ Cart Service

Responsibility

Temporary cart storage
Add / remove items

Price calculation (not final)

Why Redis?

Cart is short-lived
High read/write

In [ ]:
4️⃣ Order Service (VERY IMPORTANT)

Responsibility

Create order
Maintain order state
Order lifecycle

CREATED → PAYMENT_PENDING → PAID → SHIPPED → DELIVERED

Database

Relational DB (transaction support)

In [ ]:
5️⃣ Payment Service

Responsibility

Integrate with payment gateway

Handle callbacks (success/failure)

In [ ]:
6️⃣ Inventory Service

Responsibility

Stock management

Reserve / release stock

Important Concept
👉 Stock reservation (to avoid overselling)

In [ ]:
7️⃣ Notification Service

Responsibility

Email / SMS / Push

Order confirmation

Shipment updates

In [ ]:
4️⃣ How Request Flows (End-to-End Scenario)

🛒 Scenario 1: Place an Order
Step-by-step Flow

1️⃣ User clicks Place Order
      Frontend → API Gateway → Order Service

2️⃣ Order Service:

    Creates order with status CREATED
    Publishes event → Kafka

     ORDER_CREATED event

3️⃣ Inventory Service (Kafka consumer):

Reserves stock

Emits STOCK_RESERVED

4️⃣ Payment Service:

Processes payment

Emits PAYMENT_SUCCESS or PAYMENT_FAILED

5️⃣ Order Service:

Updates order status

6️⃣ Notification Service:

Sends email/SMS

In [ ]:
“In my E-commerce microservice architecture, each core business capability like user, 
product, order, payment, and inventory is implemented as an independent service with its own database.

Synchronous communication is used for simple queries, while Kafka is used for event-driven workflows 
such as order creation, payment processing, inventory reservation, and notifications.

We follow Saga pattern for distributed transactions and ensure loose coupling, scalability, and fault tolerance.”

In [ ]:
📌 Order Service – Low Level Design (LLD)

Order Service IS responsible for:

✔ Create order
✔ Maintain order state
✔ Orchestrate order lifecycle
✔ Publish order events
✔ Ensure idempotency

2️⃣ Core Use Cases

Create Order
Get Order by ID
Cancel Order
Update Order Status (via events)

In [ ]:
3️⃣ Order State Machine (VERY IMPORTANT)

CREATED
  ↓
PAYMENT_PENDING
  ↓
PAID
  ↓
SHIPPED
  ↓
DELIVERED

Failure paths:
PAYMENT_FAILED → CANCELLED


In [ ]:
4️⃣ Database Design (Order Service)


In [ ]:
4.1 orders table
CREATE TABLE orders (
    id              UUID PRIMARY KEY,
    user_id         UUID NOT NULL,
    total_amount    DECIMAL(10,2),
    status          VARCHAR(30),
    created_at      TIMESTAMP,
    updated_at      TIMESTAMP
);


4.2 order_items table

CREATE TABLE order_items (
    id          UUID PRIMARY KEY,
    order_id   UUID,
    product_id UUID,
    quantity   INT,
    price      DECIMAL(10,2),
    FOREIGN KEY (order_id) REFERENCES orders(id)
);

4.3 outbox_events table (Industry Standard)

CREATE TABLE outbox_events (
    id           UUID PRIMARY KEY,
    aggregate_id UUID,
    event_type   VARCHAR(50),
    payload      JSON,
    status       VARCHAR(20),
    created_at   TIMESTAMP
);
👉 Used for Outbox Pattern with Kafka

In [ ]:
5️⃣ REST APIs (Order Service)

POST /orders
Authorization: Bearer JWT

{
  "items": [
    { "product_id": "p1", "qty": 2 },
    { "product_id": "p2", "qty": 1 }
  ]
}
{
  "order_id": "o123",
  "status": "CREATED"
}

5.2 Get Order
GET /orders/{orderId}

5.3 Cancel Order
POST /orders/{orderId}/cancel


In [ ]:
6️⃣ Class Diagram (Conceptual)
OrderController
     |
OrderService
     |
OrderRepository
     |
Database


In [ ]:
7️⃣ Core Classes (Pseudo Code – Interview Friendly)
7.1 Order Entity

class Order {
    UUID id;
    UUID userId;
    BigDecimal totalAmount;
    OrderStatus status;
    List<OrderItem> items;
}

7.2 OrderStatus Enum
enum OrderStatus {
    CREATED,
    PAYMENT_PENDING,
    PAID,
    SHIPPED,
    DELIVERED,
    CANCELLED,
    PAYMENT_FAILED
}

7.3 OrderService

class OrderService {

    public Order createOrder(CreateOrderRequest req) {
        Order order = new Order();
        order.setStatus(CREATED);
        save(order);

        publishEvent("ORDER_CREATED", order);
        return order;
    }

    public void updateStatus(UUID orderId, OrderStatus status) {
        Order order = find(orderId);
        order.setStatus(status);
        save(order);
    }
}


In [ ]:
8️⃣ Kafka Integration (EVENT DRIVEN)

order-events
payment-events
inventory-events

Kafka enables event-driven microservices

In [ ]:
2️⃣ High-Level Kafka Architecture in E-commerce

Order Service  ──▶  Kafka  ──▶ Inventory Service
       │                    └─▶ Payment Service
       │                    └─▶ Notification Service
       │
       └─▶ Kafka (order-events)


In [ ]:
3️⃣ Kafka Core Concepts (Explain to Interviewer)

| Concept        | Meaning                        |
| -------------- | ------------------------------ |
| Producer       | Service that sends events      |
| Consumer       | Service that listens to events |
| Topic          | Category of messages           |
| Partition      | Parallelism + ordering         |
| Offset         | Message position               |
| Consumer Group | Load balancing                 |


In [ ]:
4️⃣ Kafka Topics Used in E-commerce

order-events
payment-events
inventory-events
notification-events


In [ ]:
Client
  ↓
Django Order Service (DRF)
  ↓
Kafka (order-events)
  ↓
Inventory / Payment Services


In [ ]:
order_service/
├── manage.py
├── order_service/
│   ├── settings.py
│   ├── urls.py
│   └── celery.py (optional)
├── orders/
│   ├── models.py
│   ├── serializers.py
│   ├── views.py
│   ├── kafka_producer.py
│   ├── kafka_consumer.py
│   ├── services.py
│   └── urls.py


In [ ]:
pip install django djangorestframework confluent-kafka psycopg2-binary


In [ ]:
Order Model:-

import uuid
from django.db import models

class Order(models.Model):
    STATUS_CHOICES = [
        ("CREATED", "CREATED"),
        ("PAID", "PAID"),
        ("PAYMENT_FAILED", "PAYMENT_FAILED"),
        ("CANCELLED", "CANCELLED"),
    ]

    id = models.UUIDField(primary_key=True, default=uuid.uuid4, editable=False)
    user_id = models.UUIDField()
    total_amount = models.DecimalField(max_digits=10, decimal_places=2)
    status = models.CharField(max_length=30, choices=STATUS_CHOICES)
    created_at = models.DateTimeField(auto_now_add=True)


In [ ]:
4️⃣ Serializer (serializers.py)
from rest_framework import serializers
from .models import Order

class OrderSerializer(serializers.ModelSerializer):
    class Meta:
        model = Order
        fields = "__all__"


In [ ]:
5️⃣ Kafka Producer (kafka_producer.py)

import json
import uuid
from confluent_kafka import Producer

conf = {
    "bootstrap.servers": "localhost:9092"
}

producer = Producer(conf)

def publish_event(topic, event_type, payload):
    event = {
        "event_id": str(uuid.uuid4()),
        "event_type": event_type,
        "payload": payload
    }

    producer.produce(
        topic=topic,
        key=str(payload.get("order_id")),
        value=json.dumps(event)
    )
    producer.flush()


In [ ]:
6️⃣ Business Logic Layer (services.py)

from .models import Order
from .kafka_producer import publish_event

def create_order(user_id, amount):
    order = Order.objects.create(
        user_id=user_id,
        total_amount=amount,
        status="CREATED"
    )

    publish_event(
        topic="order-events",
        event_type="ORDER_CREATED",
        payload={
            "order_id": str(order.id),
            "user_id": str(user_id),
            "amount": float(amount)
        }
    )

    return order


In [ ]:
7️⃣ API View (views.py)

from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from .serializers import OrderSerializer
from .services import create_order

class CreateOrderView(APIView):

    def post(self, request):
        user_id = request.data.get("user_id")
        amount = request.data.get("amount")

        order = create_order(user_id, amount)
        serializer = OrderSerializer(order)

        return Response(serializer.data, status=status.HTTP_201_CREATED)


In [ ]:
urls.py
from django.urls import path
from .views import CreateOrderView

urlpatterns = [
    path("orders/", CreateOrderView.as_view()),
]

In [ ]:
9️⃣ Kafka Consumer (Payment / Inventory Events)

kafka_consumer.py

import json
import django
django.setup()

from confluent_kafka import Consumer
from orders.models import Order

conf = {
    "bootstrap.servers": "localhost:9092",
    "group.id": "order-service-group",
    "auto.offset.reset": "earliest"
}

consumer = Consumer(conf)
consumer.subscribe(["payment-events"])

while True:
    msg = consumer.poll(1.0)

    if msg is None:
        continue
    if msg.error():
        print(msg.error())
        continue

    event = json.loads(msg.value())

    if event["event_type"] == "PAYMENT_SUCCESS":
        Order.objects.filter(
            id=event["payload"]["order_id"]
        ).update(status="PAID")

    elif event["event_type"] == "PAYMENT_FAILED":
        Order.objects.filter(
            id=event["payload"]["order_id"]
        ).update(status="PAYMENT_FAILED")


In [ ]:
                    ┌──────────────────────┐
                    │     Frontend (UI)     │
                    │  Web / Mobile App    │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │      API Gateway     │
                    │ Auth | Rate Limit    │
                    └──────────┬───────────┘
                               │
        ┌──────────────────────┼────────────────────────┐
        │                      │                        │
        ▼                      ▼                        ▼
┌──────────────┐     ┌────────────────┐       ┌─────────────────┐
│ User Service │     │ Product Service│       │  Cart Service   │
│ Auth, Profile│     │ Catalog, Price │       │ Redis (Cache)  │
└──────────────┘     └────────────────┘       └─────────────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │     Order Service    │
                    │  Order Lifecycle    │
                    │  Saga Orchestrator  │
                    └──────────┬───────────┘
                               │
                 (ORDER_CREATED event)
                               │
                               ▼
        ┌────────────────────────────────────────────────┐
        │                    KAFKA                       │
        │                                                │
        │  Topics:                                      │
        │   - order-events                              │
        │   - payment-events                            │
        │   - inventory-events                          │
        │   - notification-events                       │
        └──────────┬───────────────┬───────────────┬───┘
                   │               │               │
                   ▼               ▼               ▼
        ┌────────────────┐ ┌────────────────┐ ┌──────────────────┐
        │ Inventory Svc  │ │ Payment Service│ │ Notification Svc │
        │ Stock Reserve  │ │ Pay Gateway    │ │ Email / SMS      │
        └───────┬────────┘ └───────┬────────┘ └─────────┬────────┘
                │                  │                    │
   (STOCK_EVENT)│   (PAYMENT_EVENT)│     (NOTIFY_EVENT) │
                └──────────┬───────┴──────────┬─────────┘
                           ▼                  ▼
                    ┌────────────────────────────────┐
                    │        Order Service            │
                    │ Update Order Status             │
                    └────────────────────────────────┘


1️⃣ User Places Order

  UI → API Gateway → Order Service
  Gateway validates JWT
  Routes request

2️⃣ Order Service (VERY IMPORTANT ROLE)

    Order Service IS responsible for:
    ✔ Creating order
    ✔ Maintaining order state
    ✔ Orchestrating workflow
    ✔ Publishing events

3️⃣ Kafka – Event Backbone
    ORDER_CREATED → order-events topic

4️⃣ Inventory Service (Kafka Consumer)

Consumes ORDER_CREATED

Reserves stock

Publishes:

STOCK_RESERVED or STOCK_FAILED → inventory-events

5️⃣ Payment Service (Kafka Consumer)

Consumes ORDER_CREATED

Calls payment gateway

Publishes:
   PAYMENT_SUCCESS or PAYMENT_FAILED → payment-events

6️⃣ Order Service Listens Back

Consumes:

    inventory-events
    payment-events

Updates order status:

CREATED → PAID → CONFIRMED
or
CREATED → PAYMENT_FAILED → CANCELLED

7️⃣ Notification Service

ORDER_CONFIRMED / ORDER_CANCELLED

Sends:

Email

SMS

Push notification

| Topic               | Producer      | Consumers          |
| ------------------- | ------------- | ------------------ |
| order-events        | Order Service | Inventory, Payment |
| inventory-events    | Inventory     | Order              |
| payment-events      | Payment       | Order              |
| notification-events | Order         | Notification       |

    Kafka enables asynchronous, event-driven communication, ensuring loose coupling, scalability, and fault tolerance in E-commerce workflows.

In [ ]:
STEP 1️⃣ User Places Order

    Client → API Gateway → Order Service

Order Service:

    Validates request
    Saves order in DB
    Status = CREATED

STEP 2️⃣ Order Service Publishes Event

   Order Service
   |
   |  (publish event)
   v
Kafka Topic (order.created)


In [ ]:
🔥 NOW THE MAGIC STARTS (ASYNC)
Kafka → Inventory Service

Order Service (Producer)
        |
        |  ORDER_CREATED event
        v
   Kafka Topic (order.created)
        |
        v
Inventory Service (Consumer)

Inventory Service:
    Consumes ORDER_CREATED
    Reserves stock
    Publishes result

STOCK_RESERVED
or
STOCK_FAILED

➡️ Published to inventory-events

What Inventory Service Does as a Consumer

When Inventory Service consumes the message:

1️⃣ Reads ORDER_CREATED event
2️⃣ Validates stock
3️⃣ Reserves or reduces stock
4️⃣ Updates Inventory DB
5️⃣ Publishes a new event

In [ ]:
STEP 4️⃣ Payment Service Reacts (CONSUMER)

Kafka → Payment Service


In [ ]:
| Step             | Who calls?        | Type  |
| ---------------- | ----------------- | ----- |
| Create order     | Client → Order    | REST  |
| Publish event    | Order → Kafka     | ASYNC |
| Inventory reacts | Kafka → Inventory | EVENT |
| Payment reacts   | Kafka → Payment   | EVENT |
| Status update    | Kafka → Order     | EVENT |
